## Environment setup

#### Python interpreter location

In [ ]:
# Catat python interpreter yang dipakai nantinya saat eksekusi skrip training train_vastai.py
import sys
print(sys.executable)

#### Install Packages

In [ ]:
!pip install torch opencv-python numpy supervision rfdetr gdown "rfdetr[train,loggers]" tqdm Pillow pandas
%pip install torch opencv-python numpy supervision rfdetr gdown "rfdetr[train,loggers]" tqdm Pillow pandas

#### Setting output directory and config

In [ ]:
import os
import torch
import gdown
import zipfile

DATASET_FILE_ID = "1R5B4jABrOQHaSp-tMrTT4rsYWYtpdrFH"

ROOT_DIR = os.getcwd()
DATASET_DIR = os.path.join(ROOT_DIR, "dataset")
OUTPUT_DIR = os.path.join(ROOT_DIR, "weights")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset Directory: {DATASET_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

%cd $ROOT_DIR

#### Download dataset

In [ ]:
zip_path = os.path.join(DATASET_DIR, "dataset.zip")

print("Downloading dataset...")
gdown.download(id=DATASET_FILE_ID, output=zip_path, quiet=False)

print("\nExtracting dataset...")
with zipfile.ZipFile(file=zip_path, mode='r') as zip_ref:
    zip_ref.extractall(DATASET_DIR)

print("Cleaning up zip file...")
os.remove(zip_path)
print("Dataset is ready!")

### Edit RF-DETR library to allow change the num_queries. !! This is a vibed code !!

Default num_queries pada RF-DETR adalah 300, kode dibawah perlu dieksekusi untuk meningkatkan num_queries menjadi 500

Sumber problem: \
https://github.com/roboflow/rf-detr/issues/674 \
https://github.com/roboflow/rf-detr/issues/674#issuecomment-4190200230

In [ ]:
from pathlib import Path
import importlib
import rfdetr

pkg_root = Path(rfdetr.__file__).resolve().parent

OLD_BLOCK_DETR = '''for name in list(checkpoint["model"].keys()):
        if any(name.endswith(x) for x in query_param_names):
            checkpoint["model"][name] = checkpoint["model"][name][:num_desired_queries]'''

NEW_BLOCK_DETR = '''for name in list(checkpoint["model"].keys()):
        if any(name.endswith(x) for x in query_param_names):
            checkpoint_tensor = checkpoint["model"][name]
            checkpoint_queries = checkpoint_tensor.shape[0]
            if checkpoint_queries > num_desired_queries:
                checkpoint["model"][name] = checkpoint_tensor[:num_desired_queries]
            elif checkpoint_queries < num_desired_queries:
                # Keep checkpoint values for existing slots and preserve model
                # initialization for newly added query slots.
                model_tensor = nn_model.state_dict().get(name)
                if model_tensor is None:
                    del checkpoint["model"][name]
                    continue
                expanded = model_tensor.clone()
                expanded[:checkpoint_queries] = checkpoint_tensor
                checkpoint["model"][name] = expanded'''

OLD_BLOCK_WEIGHTS = '''for name in list(checkpoint["model"].keys()):
        if any(name.endswith(param_name) for param_name in query_param_names):
            checkpoint["model"][name] = checkpoint["model"][name][:num_desired_queries]'''

NEW_BLOCK_WEIGHTS = '''for name in list(checkpoint["model"].keys()):
        if any(name.endswith(param_name) for param_name in query_param_names):
            checkpoint_tensor = checkpoint["model"][name]
            checkpoint_queries = checkpoint_tensor.shape[0]
            if checkpoint_queries > num_desired_queries:
                checkpoint["model"][name] = checkpoint_tensor[:num_desired_queries]
            elif checkpoint_queries < num_desired_queries:
                # Keep checkpoint values for existing slots and preserve model
                # initialization for newly added query slots.
                model_tensor = nn_model.state_dict().get(name)
                if model_tensor is None:
                    del checkpoint["model"][name]
                    continue
                expanded = model_tensor.clone()
                expanded[:checkpoint_queries] = checkpoint_tensor
                checkpoint["model"][name] = expanded'''

OLD_BLOCK_MODULE = '''for name in list(checkpoint["model"].keys()):
            if any(name.endswith(x) for x in query_param_names):
                checkpoint["model"][name] = checkpoint["model"][name][:num_desired_queries]'''

NEW_BLOCK_MODULE = '''for name in list(checkpoint["model"].keys()):
            if any(name.endswith(x) for x in query_param_names):
                checkpoint_tensor = checkpoint["model"][name]
                checkpoint_queries = checkpoint_tensor.shape[0]
                if checkpoint_queries > num_desired_queries:
                    checkpoint["model"][name] = checkpoint_tensor[:num_desired_queries]
                elif checkpoint_queries < num_desired_queries:
                    # Keep checkpoint values for existing slots and preserve model
                    # initialization for newly added query slots.
                    model_tensor = self.model.state_dict().get(name)
                    if model_tensor is None:
                        del checkpoint["model"][name]
                        continue
                    expanded = model_tensor.clone()
                    expanded[:checkpoint_queries] = checkpoint_tensor
                    checkpoint["model"][name] = expanded'''

PATCH_SPECS = [
    ("detr.py", OLD_BLOCK_DETR, NEW_BLOCK_DETR),
    ("models/weights.py", OLD_BLOCK_WEIGHTS, NEW_BLOCK_WEIGHTS),
    ("training/module_model.py", OLD_BLOCK_MODULE, NEW_BLOCK_MODULE),
]

def apply_text_patch(file_path: Path, old_block: str, new_block: str) -> str:
    text = file_path.read_text(encoding="utf-8")

    if new_block in text:
        return "already_patched"

    if old_block not in text:
        return "pattern_not_found"

    backup_path = file_path.with_suffix(file_path.suffix + ".bak")
    if not backup_path.exists():
        backup_path.write_text(text, encoding="utf-8")

    updated = text.replace(old_block, new_block, 1)
    file_path.write_text(updated, encoding="utf-8")
    return "patched"

results = {}
for relative_path, old_block, new_block in PATCH_SPECS:
    target = pkg_root / relative_path
    if not target.exists():
        results[relative_path] = "file_not_found"
        continue
    results[relative_path] = apply_text_patch(target, old_block, new_block)

print("RF-DETR patch summary:")
for relative_path, status in results.items():
    print(f"- {relative_path}: {status}")

failed = [p for p, s in results.items() if s in {"file_not_found", "pattern_not_found"}]
if failed:
    raise RuntimeError(
        "Patch was not fully applied. This may be a different rfdetr version. "
        f"Check these files: {failed}"
    )

import rfdetr.detr
import rfdetr.models.weights
import rfdetr.training.module_model
importlib.reload(rfdetr.detr)
importlib.reload(rfdetr.models.weights)
importlib.reload(rfdetr.training.module_model)

print("Patch applied successfully. You can now run RFDETRSize(num_queries=500, num_select=500).")

In [ ]:
# Check apakah num_queries telah menjadi 500
from rfdetr import RFDETRSmall # ubah model yang diinginkan, misal RFDETRMedium

model = RFDETRSmall(num_queries=500, num_select=500)

# Verify the num_queries now to be 500 instead of the default 300
print(f"Target Configuration Queries: {model.model_config.num_queries}")
print(f"Target Postprocess Select: {model.model_config.num_select}")

if hasattr(model.model, "args") and hasattr(model.model.args, "num_queries"):
    print(f"Underlying PyTorch Architecture Queries: {model.model.args.num_queries}")
else:
    print("Could not find underlying args, but configuration is set.")

if hasattr(model.model.model, "query_feat"):
    print(f"query_feat weight shape: {tuple(model.model.model.query_feat.weight.shape)}")